# Prognozowanie cen nieruchomości w Kalifornii – case study

<div style="text-align: center;"><img src=".\\Images\\California_Housing.png" alt="California housing" width="400" height="120" style="margin: 10px;" /></div>

---

**Cel projektu**

Wzrost cen nieruchomości i presja mieszkaniowa to kluczowe wyzwania w stanie Kalifornia, szczególnie w obszarach metropolitalnych. Prognozowanie mediany ceny domów na podstawie lokalnych cech może pomóc:
- urbanistom w planowaniu rozwoju infrastruktury,
- inwestorom w podejmowaniu decyzji,
- władzom lokalnym w projektowaniu polityki mieszkaniowej.

Celem projektu jest zbudowanie **modelu regresyjnego**, który na podstawie danych demograficznych i geograficznych prognozuje **mediana ceny domu (w tysiącach dolarów)** w danym regionie.

---

**Dane źródłowe**

Zbiór danych pochodzi z `sklearn.datasets.fetch_california_housing()`, a pierwotnie z **California Housing Survey (1990)**.

- Liczba rekordów: 20 640
- Liczba cech: 8
- Każdy wiersz odpowiada agregowanym danym dla małego obszaru geograficznego (block group)
- Dane zawierają m.in.:
  - średni wiek budynków,
  - średni dochód gospodarstwa domowego,
  - gęstość zaludnienia,
  - położenie geograficzne (szerokość, długość geograficzna),
  - liczba pokoi, sypialni itd.

---

**Zmienna docelowa**

- `MedHouseVal` – **mediana wartości domów** (w dziesiątkach tysięcy dolarów)
- Zadanie: regresja ciągła

---

**Zadanie**

Zbadaj dla zbioru zastosowanie metod zespołowych (analigiczny zestaw co w materiałach):
- Bagging
- Boosting
- Stacking

Jakość predykcji oceń za pomocą:
- R² (coefficient of determination),
- MAE (mean absolute error),
- RMSE (root mean squared error).
     
---

# Rozwiązanie

<span style="color: navy; font-weight: bold;">Wczytanie danych i podział na zbiory treningowy i testowy</span>

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)        # pokazuj wszystkie kolumny
pd.set_option('display.expand_frame_repr', False) # nie łam wierszy na kilka linii

# Dane
X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

<span style="color: red; font-weight: bold;">Bagging</span>

In [12]:

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

models_bag = {
    "Decision Tree":  DecisionTreeRegressor(random_state=42),
    "Bagging":        BaggingRegressor(estimator=DecisionTreeRegressor(), n_estimators=50, random_state=42),
    "Random Forest":  RandomForestRegressor(n_estimators=50, random_state=42),
    "Extra Trees":    ExtraTreesRegressor(n_estimators=50, random_state=42),
}

results_bag = []
for name, model in models_bag.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results_bag.append({
        "Model": name,
        "R2":    round(r2_score(y_test, y_pred), 4),
        "MAE":   round(mean_absolute_error(y_test, y_pred), 4),
        "RMSE":  round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
    })

pd.DataFrame(results_bag)

,Model,R2,MAE,RMSE
0,Decision Tree,0.6006,0.4671,0.7270
1,Bagging,0.8048,0.3327,0.5082
2,Random Forest,0.8053,0.3323,0.5076
3,Extra Trees,0.8102,0.3268,0.5011


<span style="color: red; font-weight: bold;">Boosting</span>

In [13]:

from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

models_boost = {
    "Decision Tree":        DecisionTreeRegressor(random_state=42),
    "AdaBoost":             AdaBoostRegressor(n_estimators=50, random_state=42),
    "Gradient Boosting":    GradientBoostingRegressor(n_estimators=100, random_state=42),
    "HistGradient Boosting": HistGradientBoostingRegressor(random_state=42),
}

results_boost = []
for name, model in models_boost.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results_boost.append({
        "Model": name,
        "R2":    round(r2_score(y_test, y_pred), 4),
        "MAE":   round(mean_absolute_error(y_test, y_pred), 4),
        "RMSE":  round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
    })

pd.DataFrame(results_boost)


,Model,R2,MAE,RMSE
0,Decision Tree,0.6006,0.4671,0.7270
1,AdaBoost,0.3808,0.7918,0.9052
2,Gradient Boosting,0.7812,0.3712,0.5381
3,HistGradient Boosting,0.8364,0.3120,0.4652


<span style="color: red; font-weight: bold;">Stacking</span>

In [11]:

from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

base_learners = [
    ("dt",  DecisionTreeRegressor(random_state=42)),
    ("rf",  RandomForestRegressor(n_estimators=50, random_state=42)),
    ("hgb", HistGradientBoostingRegressor(random_state=42)),
]

stack_model = StackingRegressor(estimators=base_learners, final_estimator=Ridge())

models_stack = {
    "Random Forest":          RandomForestRegressor(n_estimators=50, random_state=42),
    "Stacking (dt, rf, hgb)": stack_model,
}

results_stack = []
for name, model in models_stack.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results_stack.append({
        "Model": name,
        "R2":    round(r2_score(y_test, y_pred), 4),
        "MAE":   round(mean_absolute_error(y_test, y_pred), 4),
        "RMSE":  round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
    })

pd.DataFrame(results_stack)


,Model,R2,MAE,RMSE
0,Random Forest,0.8053,0.3323,0.5076
1,"Stacking (dt, rf, hgb)",0.8366,0.3099,0.4649


<span style="color: red; font-weight: bold;">Podsumuj modele</span>

In [14]:

# Zestawienie wszystkich modeli – sortowanie po R² (im wyższy, tym lepiej)

all_results = results_bag + results_boost + results_stack

summary_df = (
    pd.DataFrame(all_results)
    .drop_duplicates(subset="Model")
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

summary_df


,Model,R2,MAE,RMSE
0,"Stacking (dt, rf, hgb)",0.8366,0.3099,0.4649
1,HistGradient Boosting,0.8364,0.3120,0.4652
2,Extra Trees,0.8102,0.3268,0.5011
3,Random Forest,0.8053,0.3323,0.5076
4,Bagging,0.8048,0.3327,0.5082
5,Gradient Boosting,0.7812,0.3712,0.5381
6,Decision Tree,0.6006,0.4671,0.7270
7,AdaBoost,0.3808,0.7918,0.9052


In [16]:

# Wnioski – tabela posortowana po R²; im wyżej, tym lepszy model
summary_df


,Model,R2,MAE,RMSE
0,"Stacking (dt, rf, hgb)",0.8366,0.3099,0.4649
1,HistGradient Boosting,0.8364,0.3120,0.4652
2,Extra Trees,0.8102,0.3268,0.5011
3,Random Forest,0.8053,0.3323,0.5076
4,Bagging,0.8048,0.3327,0.5082
5,Gradient Boosting,0.7812,0.3712,0.5381
6,Decision Tree,0.6006,0.4671,0.7270
7,AdaBoost,0.3808,0.7918,0.9052
